[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/13_dense_sparse_late_interaction/13_proyecto_dense_sparse_late_interaction.ipynb)

# Proyecto 13 · Retrieval híbrido de Políglota: fusión y reranking

**Nivel:** 🟠 Avanzado · **Duración:** ≈ 2 h · **GPU:** T4 opcional (CPU basta con `FAST_DEV_RUN = True`) · **Cómputo:** ≈ 0–1 unidades de Colab.

**Conexión.** *Viene de:* módulo 12 (el embedder denso y su configuración) y de la lección 13. *Va hacia:* módulo 14 (indexar los vectores de este sistema con ANN) y 15 (evaluarlo con intervalos de confianza).

## 1 · Contexto de negocio
El CTO de **Políglota Labs** quiere pasar de «un embedder» a un **sistema de recuperación híbrido**: BM25 + denso multilingüe + late interaction, fusionados, con un **cross-encoder** que reordena el top-K bajo un **presupuesto de latencia** (≈ 200 ms por consulta para la etapa de reranking). Tu trabajo: implementar las piezas, elegir la fusión **sin hacer trampa con el tuning** y entregar el pipeline con su análisis de coste/beneficio.

## 2 · Dataset
Dos bancos de pruebas (ver lección): el **corpus canónico** del módulo 00 (`carga_corpus`; sintético con `⚠️` si no hay acceso) y el **mini golden set de paráfrasis ES↔EN↔código** (58 queries).

## 3 · Entregables
1. `BM25` propio (verificado contra `rank_bm25`), `maxsim_matriz` vectorizado (verificado contra la fórmula ingenua).
2. `rrf`, `lineal` y `alfa_cv` (α elegido por validación cruzada de 2 pliegues).
3. `hibrido_rerank`: pipeline fusión → cross-encoder sobre el top-K, y `elige_K` que respeta el presupuesto de latencia.
4. Tabla final de nDCG@10 por retriever, fusión y pipeline en **ambos** bancos + una conclusión escrita.

## 4 · Rúbrica (umbrales calibrados con lo que MIDE la solución de referencia)
Los umbrales son **invariantes relativas** a tu propia ejecución. La solución de referencia se ejecutó en un entorno **sin acceso a Hugging Face** (corpus sintético, embedder denso de respaldo TF-IDF+SVD, ColBERT de juguete y reranker de respaldo) y con modelos diminutos de pesos aleatorios para ejercitar las rutas de `sentence-transformers`; **no se verificó con modelos reales**. Valores medidos en la última columna.

| # | Criterio | Umbral | Medido por la solución |
|---|---|---|---|
| R1 | BM25 propio (IDF `okapi`) coincide con `rank_bm25` | `allclose` | True |
| R2 | `maxsim_matriz` coincide con la fórmula ingenua en 20 pares | atol 1e-4 | True |
| R3 | RRF reproduce el ejemplo a mano (rangos 1 y 3 → 1/61 + 1/63) | exacto | True |
| R4 | Canónico: mejor fusión ≥ mejor retriever individual − 0.03 nDCG@10 | cumplido | fusión 0.848 vs individual 0.861 (Δ = −0.013) |
| R5 | Paráfrasis: mejor fusión ≥ mejor retriever individual − 0.03 nDCG@10 | cumplido | fusión 0.318 vs individual 0.320 (Δ = −0.002) |
| R6 | El nDCG de `alfa_cv` (validación cruzada) no supera al de α elegido en las mismas queries (sin fuga) | `cv ≤ oráculo + 1e-9` | 0.848 ≤ 0.861 y 0.318 ≤ 0.323 |
| R7 | `hibrido_rerank` devuelve ≥ 10 documentos por consulta y solo reordena el top-K | cumplido | True (≥ 10 en todas las queries) |
| R8 | `elige_K` devuelve el mayor K con latencia ≤ presupuesto | cumplido | True |
| R9 | Conclusión escrita con ≥ 3 números de tus tablas | — | revisión manual |

## 5 · Preparación (celdas dadas)

In [ ]:
!pip install -q -U sentence-transformers transformers scikit-learn scipy matplotlib pandas rank_bm25

In [ ]:
import os, sys, time, math, random, json, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU en minutos). Pon False en Colab con GPU para la escala completa.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))

try:
    import torch
    torch.manual_seed(SEED)
    torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:      # los fragmentos "desde cero" de NumPy funcionan sin torch
    torch, device = None, "cpu"
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
def ndcgs_por_query(run: dict, qrels: dict, k: int = 10):
    """nDCG@k de cada query (mismo filtro que evaluate(): solo queries con algún relevante). Sirve para bootstrap."""
    return np.array([ndcg_at_k(run.get(q, []), r, k) for q, r in qrels.items() if any(v > 0 for v in r.values())])

In [ ]:
# Mini-corpus sintético "Políglota": 29 conceptos técnicos x {EN, ES, código}. Es el FALLBACK del cargador de datos;
# tiene el mismo esquema que el corpus real. Las queries están parafraseadas a propósito para que no compartan
# casi ninguna palabra con el documento (así BM25 no es trivial y los embeddings multilingües sí aportan).
PARAFRASIS = [
 ("hnsw",
  "HNSW builds a multi-layer proximity graph. A search enters at the top layer, walks greedily towards the target and descends layer by layer, returning approximate neighbors in logarithmic time.",
  "HNSW construye un grafo de proximidad con varias capas. La búsqueda entra por la capa superior, avanza de forma voraz hacia el objetivo y desciende capa a capa, devolviendo vecinos aproximados en tiempo logarítmico.",
  "def greedy_search(graph, vectors, q, entry):\n    best, improved = entry, True\n    while improved:\n        improved = False\n        for nb in graph[best]:\n            if dist(vectors[nb], q) < dist(vectors[best], q):\n                best, improved = nb, True\n    return best",
  "find similar vectors fast by hopping through a layered network", "buscar vectores parecidos rápido saltando por una red de niveles"),
 ("inverted_index",
  "An inverted index maps every term to the list of documents that contain it, so a keyword query only reads the posting lists of its terms instead of scanning the whole collection.",
  "Un índice invertido asocia cada término con la lista de documentos que lo contienen, de modo que una consulta solo lee las listas de sus términos en lugar de recorrer toda la colección.",
  "def build_index(docs):\n    index = {}\n    for doc_id, text in docs.items():\n        for term in set(text.lower().split()):\n            index.setdefault(term, []).append(doc_id)\n    return index",
  "data structure that avoids scanning every document for a word", "estructura que evita recorrer todos los textos al buscar una palabra"),
 ("bm25",
  "BM25 scores a document by summing, for each query term, its inverse document frequency times a saturating term frequency, normalized by document length with parameters k1 and b.",
  "BM25 puntúa un documento sumando, para cada término de la consulta, su frecuencia inversa de documento por una frecuencia de término saturada, normalizada por la longitud del documento con los parámetros k1 y b.",
  "def bm25_term(tf, idf, dl, avgdl, k1=1.2, b=0.75):\n    norm = k1 * (1 - b + b * dl / avgdl)\n    return idf * tf * (k1 + 1) / (tf + norm)",
  "classic keyword ranking formula with length normalization", "fórmula clásica de ranking por palabras clave que penaliza textos largos"),
 ("bpe",
  "Byte-pair encoding starts from single bytes and repeatedly merges the most frequent adjacent pair into a new token until the vocabulary reaches the desired size.",
  "La codificación por pares de bytes parte de bytes sueltos y fusiona repetidamente el par adyacente más frecuente en un nuevo token hasta alcanzar el tamaño de vocabulario deseado.",
  "def most_frequent_pair(ids):\n    counts = {}\n    for a, b in zip(ids, ids[1:]):\n        counts[(a, b)] = counts.get((a, b), 0) + 1\n    return max(counts, key=counts.get)",
  "how a language model splits text into subword pieces by merging common pairs", "cómo un modelo de lenguaje trocea el texto en piezas fusionando pares comunes"),
 ("cosine",
  "Cosine similarity measures the angle between two vectors: the dot product divided by the product of their norms. It ignores magnitude and ranges from -1 to 1.",
  "La similitud del coseno mide el ángulo entre dos vectores: el producto punto dividido por el producto de sus normas. Ignora la magnitud y va de -1 a 1.",
  "def cosine(a, b):\n    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))",
  "compare two vectors by direction rather than length", "comparar dos vectores por su dirección y no por su longitud"),
 ("kmeans",
  "K-means alternates two steps: assign every point to its nearest centroid, then move each centroid to the mean of its assigned points, until assignments stop changing.",
  "K-means alterna dos pasos: asignar cada punto a su centroide más cercano y mover cada centroide a la media de sus puntos, hasta que las asignaciones dejan de cambiar.",
  "def kmeans_step(X, C):\n    d = ((X[:, None] - C[None]) ** 2).sum(-1)\n    a = d.argmin(1)\n    return np.stack([X[a == k].mean(0) for k in range(len(C))]), a",
  "group unlabeled points into clusters around moving centers", "agrupar puntos sin etiquetas alrededor de centros que se desplazan"),
 ("pq",
  "Product quantization splits a vector into sub-vectors and replaces each by the index of its nearest codeword, compressing a float vector to a few bytes while allowing fast approximate distances.",
  "La cuantización de producto divide un vector en subvectores y sustituye cada uno por el índice de su palabra de código más cercana, comprimiendo un vector de floats a pocos bytes con distancias aproximadas rápidas.",
  "def pq_encode(x, codebooks):\n    subs = np.split(x, len(codebooks))\n    return [int(((cb - s) ** 2).sum(1).argmin()) for s, cb in zip(subs, codebooks)]",
  "shrink embeddings to a handful of bytes using small codebooks", "reducir embeddings a unos pocos bytes usando diccionarios pequeños"),
 ("reranker",
  "A cross-encoder reranker reads the query and a candidate document together in one transformer pass and outputs a relevance score; it is accurate but too slow for the whole corpus.",
  "Un reranker cross-encoder lee la consulta y un documento candidato juntos en una sola pasada del transformer y devuelve una puntuación de relevancia; es preciso pero demasiado lento para todo el corpus.",
  "def rerank(model, query, candidates, top_n=10):\n    scores = model.predict([(query, c) for c in candidates])\n    order = np.argsort(-scores)[:top_n]\n    return [candidates[i] for i in order]",
  "second-stage model that rescores a short list of candidates jointly with the question", "modelo de segunda etapa que reordena una lista corta mirando pregunta y texto a la vez"),
 ("matryoshka",
  "Matryoshka representation learning trains an embedding so that its first m dimensions already form a good smaller embedding, letting you truncate vectors and re-normalize them to save memory.",
  "El aprendizaje de representaciones Matryoshka entrena un embedding para que sus primeras m dimensiones ya formen un embedding más pequeño útil, permitiendo truncar los vectores y renormalizarlos para ahorrar memoria.",
  "def truncate(emb, m):\n    out = emb[:, :m]\n    return out / np.linalg.norm(out, axis=1, keepdims=True)",
  "cut a vector to fewer coordinates without retraining", "recortar un vector a menos coordenadas sin volver a entrenar"),
 ("contrastive",
  "Contrastive loss pulls the embeddings of matching pairs together and pushes apart the other examples in the batch, using a softmax over similarities scaled by a temperature.",
  "La pérdida contrastiva acerca los embeddings de los pares que coinciden y aleja el resto de ejemplos del lote, usando un softmax sobre similitudes escaladas por una temperatura.",
  "def info_nce(q, d, tau=0.05):\n    logits = (q @ d.T) / tau\n    labels = torch.arange(len(q))\n    return torch.nn.functional.cross_entropy(logits, labels)",
  "training objective that treats other items in the batch as wrong answers", "objetivo de entrenamiento que trata los demás ejemplos del lote como respuestas erróneas"),
 ("attention",
  "Self-attention lets every token build its representation as a weighted average of all value vectors, with weights given by the softmax of scaled query-key dot products.",
  "La autoatención permite que cada token construya su representación como un promedio ponderado de todos los vectores de valor, con pesos dados por el softmax de productos punto consulta-clave escalados.",
  "def attention(q, k, v):\n    w = torch.softmax(q @ k.T / q.shape[-1] ** 0.5, dim=-1)\n    return w @ v",
  "mechanism where each word looks at all the other words to update itself", "mecanismo en el que cada palabra mira a todas las demás para actualizarse"),
 ("cosine_decay",
  "A cosine learning-rate schedule starts at the peak value after warmup and decays smoothly towards zero following half a cosine wave, avoiding abrupt drops late in training.",
  "Un calendario de tasa de aprendizaje con coseno parte del valor máximo tras el calentamiento y decae suavemente hacia cero siguiendo media onda de coseno, evitando caídas bruscas al final del entrenamiento.",
  "def lr_at(step, total, peak=3e-4):\n    return 0.5 * peak * (1 + math.cos(math.pi * step / total))",
  "how to shrink the step size smoothly while a network trains", "cómo reducir poco a poco el tamaño del paso mientras una red entrena"),
 ("dropout",
  "Dropout randomly zeroes a fraction of activations during training and rescales the rest, which discourages co-adaptation of neurons and acts as a regularizer; it is disabled at inference.",
  "Dropout pone a cero al azar una fracción de las activaciones durante el entrenamiento y reescala el resto, lo que evita la coadaptación de neuronas y actúa como regularizador; se desactiva en inferencia.",
  "def dropout(x, p=0.1, training=True):\n    if not training:\n        return x\n    mask = (np.random.rand(*x.shape) > p) / (1 - p)\n    return x * mask",
  "randomly switching off neurons to reduce overfitting", "apagar neuronas al azar para reducir el sobreajuste"),
 ("layernorm",
  "Layer normalization standardizes each token's feature vector to zero mean and unit variance and then applies a learned scale and shift, stabilizing the training of deep transformers.",
  "La normalización por capas estandariza el vector de características de cada token a media cero y varianza uno y luego aplica una escala y un desplazamiento aprendidos, estabilizando el entrenamiento de transformers profundos.",
  "def layer_norm(x, g, b, eps=1e-5):\n    mu = x.mean(-1, keepdims=True)\n    var = x.var(-1, keepdims=True)\n    return g * (x - mu) / np.sqrt(var + eps) + b",
  "rescale each feature vector so deep networks train stably", "reescalar cada vector de características para que las redes profundas entrenen estables"),
 ("grad_clip",
  "Gradient clipping rescales the gradient whenever its global norm exceeds a threshold, which prevents rare huge updates from destabilizing training.",
  "El recorte de gradiente reescala el gradiente cada vez que su norma global supera un umbral, lo que evita que actualizaciones enormes y poco frecuentes desestabilicen el entrenamiento.",
  "def clip_grad(grads, max_norm=1.0):\n    total = math.sqrt(sum(float((g ** 2).sum()) for g in grads))\n    scale = min(1.0, max_norm / (total + 1e-6))\n    return [g * scale for g in grads]",
  "limit the size of parameter updates to avoid exploding values", "limitar el tamaño de las actualizaciones de parámetros para evitar valores que explotan"),
 ("lru_cache",
  "An LRU cache keeps the most recently used entries and evicts the least recently used one when it is full, giving constant-time lookups with a hash map plus a linked list.",
  "Una caché LRU conserva las entradas usadas más recientemente y expulsa la menos reciente cuando se llena, con búsquedas en tiempo constante mediante un mapa hash y una lista enlazada.",
  "from collections import OrderedDict\n\nclass LRU:\n    def __init__(self, cap):\n        self.cap, self.d = cap, OrderedDict()\n    def get(self, k):\n        self.d.move_to_end(k)\n        return self.d[k]",
  "memory that forgets whatever was touched longest ago", "memoria que olvida lo que se tocó hace más tiempo"),
 ("token_bucket",
  "A token bucket rate limiter adds tokens at a fixed rate up to a maximum capacity; each request consumes one token and is rejected when the bucket is empty, allowing short bursts.",
  "Un limitador de tasa de cubeta de tokens añade tokens a ritmo fijo hasta una capacidad máxima; cada petición consume un token y se rechaza si la cubeta está vacía, lo que permite ráfagas cortas.",
  "def allow(bucket, now, rate=5.0, cap=10.0):\n    bucket['tokens'] = min(cap, bucket['tokens'] + (now - bucket['t']) * rate)\n    bucket['t'] = now\n    if bucket['tokens'] >= 1:\n        bucket['tokens'] -= 1\n        return True\n    return False",
  "stop an API client from sending too many requests per second", "impedir que un cliente de una API envíe demasiadas peticiones por segundo"),
 ("binary_search",
  "Binary search finds a value in a sorted array by repeatedly comparing it with the middle element and discarding half of the remaining range, so it needs only logarithmic comparisons.",
  "La búsqueda binaria encuentra un valor en un arreglo ordenado comparándolo repetidamente con el elemento central y descartando la mitad del rango restante, por lo que solo necesita comparaciones logarítmicas.",
  "def binary_search(a, x):\n    lo, hi = 0, len(a) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if a[mid] == x:\n            return mid\n        lo, hi = (mid + 1, hi) if a[mid] < x else (lo, mid - 1)\n    return -1",
  "locate an item in an ordered list by halving the range each time", "localizar un elemento en una lista ordenada partiendo el rango a la mitad cada vez"),
 ("hash_collision",
  "A hash table maps keys to buckets with a hash function; when two keys land in the same bucket a collision occurs, resolved by chaining or open addressing.",
  "Una tabla hash asigna claves a cubetas mediante una función hash; cuando dos claves caen en la misma cubeta ocurre una colisión, que se resuelve con encadenamiento o direccionamiento abierto.",
  "def put(table, key, value):\n    bucket = table[hash(key) % len(table)]\n    for i, (k, _) in enumerate(bucket):\n        if k == key:\n            bucket[i] = (key, value)\n            return\n    bucket.append((key, value))",
  "what happens when two different keys map to the same slot", "qué pasa cuando dos claves distintas caen en la misma casilla"),
 ("deadlock",
  "A deadlock happens when two threads each hold a lock the other needs and wait forever; acquiring locks in a single global order prevents the circular wait.",
  "Un interbloqueo ocurre cuando dos hilos retienen cada uno un candado que el otro necesita y esperan para siempre; adquirir los candados en un único orden global evita la espera circular.",
  "def transfer(a, b, amount):\n    first, second = sorted([a, b], key=id)\n    with first.lock:\n        with second.lock:\n            a.balance -= amount\n            b.balance += amount",
  "two threads waiting on each other forever", "dos hilos esperándose mutuamente para siempre"),
 ("btree_index",
  "A B-tree index keeps keys sorted in a balanced tree with wide nodes, so a database can find a row or scan a range with only a few disk reads.",
  "Un índice B-tree mantiene las claves ordenadas en un árbol balanceado con nodos anchos, de modo que una base de datos puede encontrar una fila o recorrer un rango con pocas lecturas de disco.",
  "def create_index(cursor):\n    cursor.execute('CREATE INDEX idx_users_email ON users (email)')\n    cursor.execute('EXPLAIN SELECT * FROM users WHERE email = ?', ('a@b.c',))",
  "make database lookups on a column faster with a sorted tree", "acelerar las búsquedas de una columna en una base de datos con un árbol ordenado"),
 ("sharding",
  "Sharding splits a large dataset across several machines by a shard key, so each node stores and serves only part of the data and capacity grows horizontally.",
  "El sharding reparte un conjunto de datos grande entre varias máquinas según una clave de partición, de modo que cada nodo almacena y sirve solo una parte y la capacidad crece horizontalmente.",
  "def shard_for(key, n_shards):\n    h = int(hashlib.md5(key.encode()).hexdigest(), 16)\n    return h % n_shards",
  "distribute data over many servers so no single one holds everything", "distribuir los datos entre muchos servidores para que ninguno guarde todo"),
 ("backoff",
  "Exponential backoff retries a failed request after waiting 2^n times a base delay, usually with random jitter, so that many clients do not hammer a recovering service at once.",
  "El reintento con retroceso exponencial repite una petición fallida tras esperar 2^n veces un retardo base, normalmente con jitter aleatorio, para que muchos clientes no saturen a la vez un servicio que se recupera.",
  "def retry(fn, tries=5, base=0.5):\n    for n in range(tries):\n        try:\n            return fn()\n        except Exception:\n            time.sleep(base * 2 ** n * random.uniform(0.5, 1.5))\n    raise RuntimeError('agotados los reintentos')",
  "wait longer after each failed network call before trying again", "esperar más tras cada llamada de red fallida antes de volver a intentar"),
 ("mocking",
  "In unit tests a mock replaces a real dependency such as a network client, so the test is fast, deterministic and verifies how the code under test calls it.",
  "En las pruebas unitarias un mock sustituye una dependencia real, como un cliente de red, para que la prueba sea rápida, determinista y verifique cómo la llama el código bajo prueba.",
  "from unittest.mock import patch\n\ndef test_fetch():\n    with patch('app.http_get', return_value={'ok': True}) as m:\n        assert fetch('/x') == {'ok': True}\n        m.assert_called_once_with('/x')",
  "fake a service in a test so it runs without the internet", "simular un servicio en una prueba para que corra sin internet"),
 ("git_rebase",
  "Git rebase replays the commits of your branch on top of another base commit, producing a linear history instead of a merge commit, but it rewrites commit hashes.",
  "Git rebase reaplica los commits de tu rama encima de otro commit base, produciendo un historial lineal en lugar de un commit de fusión, pero reescribe los hashes de los commits.",
  "import subprocess\n\ndef rebase_onto_main():\n    subprocess.run(['git', 'fetch', 'origin'], check=True)\n    subprocess.run(['git', 'rebase', 'origin/main'], check=True)",
  "move my branch commits to the tip of main to keep a straight history", "mover los commits de mi rama a la punta de main para mantener un historial recto"),
 ("docker",
  "A Docker container packages an application with its dependencies and runs it as an isolated process that shares the host kernel, which makes deployments reproducible and lightweight.",
  "Un contenedor Docker empaqueta una aplicación con sus dependencias y la ejecuta como un proceso aislado que comparte el kernel del anfitrión, lo que hace los despliegues reproducibles y ligeros.",
  "import subprocess\n\ndef run_container(image, port=8000):\n    subprocess.run(['docker', 'run', '--rm', '-p', f'{port}:{port}', image], check=True)",
  "ship software with everything it needs so it runs the same everywhere", "entregar software con todo lo que necesita para que corra igual en cualquier sitio"),
 ("pagination",
  "Cursor-based pagination returns a page of results plus an opaque cursor pointing to the last item, which is more stable and efficient than offsets when the data changes between requests.",
  "La paginación por cursor devuelve una página de resultados junto con un cursor opaco que apunta al último elemento, algo más estable y eficiente que los desplazamientos cuando los datos cambian entre peticiones.",
  "def list_items(db, cursor=None, limit=20):\n    rows = db.query('SELECT * FROM items WHERE id > ? ORDER BY id LIMIT ?', cursor or 0, limit)\n    return rows, (rows[-1].id if rows else None)",
  "return long API results in chunks that stay consistent while data changes", "devolver resultados largos de una API en trozos que se mantienen coherentes mientras los datos cambian"),
 ("regex",
  "A regular expression describes a set of strings with literals, character classes and quantifiers; the engine scans the text and returns the matches, but nested quantifiers can cause catastrophic backtracking.",
  "Una expresión regular describe un conjunto de cadenas con literales, clases de caracteres y cuantificadores; el motor recorre el texto y devuelve las coincidencias, pero los cuantificadores anidados pueden causar retroceso catastrófico.",
  "import re\n\nEMAIL = re.compile(r'[\\w.+-]+@[\\w-]+\\.[\\w.]+')\n\ndef find_emails(text):\n    return EMAIL.findall(text)",
  "pattern language to extract pieces of text such as e-mail addresses", "lenguaje de patrones para extraer fragmentos de texto como direcciones de correo"),
 ("unicode_norm",
  "Unicode normalization converts equivalent character sequences to a canonical form: NFC composes accents into single code points, while NFD decomposes them into base letter plus combining mark.",
  "La normalización Unicode convierte secuencias de caracteres equivalentes a una forma canónica: NFC compone los acentos en un solo punto de código, mientras que NFD los descompone en letra base más marca combinante.",
  "import unicodedata\n\ndef strip_accents(s):\n    nfd = unicodedata.normalize('NFD', s)\n    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')",
  "make visually identical accented strings compare as equal", "lograr que cadenas con tildes visualmente idénticas se comparen como iguales"),
]
print(f"{len(PARAFRASIS)} conceptos x 3 documentos (en/es/código) en el mini golden set de paráfrasis")

In [ ]:
def corpus_parafrasis() -> dict:
    """Mini golden set de PARÁFRASIS y búsqueda CRUZADA ES<->EN<->código (29 conceptos x 3 documentos, 58 queries).
    Mismo esquema que carga_corpus(). Las queries casi no comparten palabras con sus documentos: BM25 sufre y los
    modelos semánticos/multilingües deberían ganar. Es un conjunto pequeño y escrito a mano: úsalo para diagnosticar,
    no para reportar cifras de producción."""
    docs, queries, qrels = [], [], {}
    for key, en, es, code, q_en, q_es in PARAFRASIS:
        ids = {"en": f"{key}-en", "es": f"{key}-es", "code": f"{key}-code"}
        docs += [{"id": ids["en"], "lang": "en", "text": en}, {"id": ids["es"], "lang": "es", "text": es},
                 {"id": ids["code"], "lang": "code", "text": code}]
        for lang, q in (("en", q_en), ("es", q_es)):
            qid = f"par-{key}-{lang}"
            queries.append({"qid": qid, "lang": lang, "text": q})
            qrels[qid] = {d: 1 for d in ids.values()}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "paraphrase-golden"}

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

class Bench:
    """Un corpus + sus queries + las matrices de scores (n_queries x n_docs) de cada retriever."""
    def __init__(self, nombre, corpus):
        self.nombre, self.corpus = nombre, corpus
        self.ids = [d["id"] for d in corpus["docs"]]
        self.txt = [d["text"] for d in corpus["docs"]]
        self.qs = corpus["queries"]
        self.qtxt = [q["text"] for q in self.qs]
        self.qrels = corpus["qrels"]
        self.S = {}                                   # nombre del retriever -> matriz de scores
    def run(self, S, k=10):
        return {q["qid"]: rank_from_scores(S[i], self.ids, k) for i, q in enumerate(self.qs)}
    def ndcg(self, S, k=10):
        return evaluate(self.run(S, k), self.qrels, k)["ndcg"]
    def ndcg_q(self, S, k=10):
        """nDCG@k por query (alineado con self.qs): sirve para validación cruzada y bootstrap."""
        run = self.run(S, k)
        return np.array([ndcg_at_k(run[q["qid"]], self.qrels[q["qid"]], k) for q in self.qs])
    def resumen(self):
        filas = []
        for nombre, S in self.S.items():
            r = evaluate(self.run(S), self.qrels)
            filas.append({"retriever": nombre, "nDCG@10": round(r["ndcg"], 3), "Recall@10": round(r["recall"], 3), "MRR@10": round(r["mrr"], 3)})
        return pd.DataFrame(filas).set_index("retriever")

CORPUS = carga_corpus(FAST_DEV_RUN)
PAR = corpus_parafrasis()
B1, B2 = Bench("canónico (Políglota)", CORPUS), Bench("paráfrasis ES↔EN↔código", PAR)
BENCHES = [B1, B2]
for b in BENCHES:
    print(f"{b.nombre}: fuente={b.corpus['source']} · {len(b.ids)} docs · {len(b.qs)} queries")

In [ ]:
# Retriever denso: el primer embedder que se pueda cargar de verdad; si no, el fallback TF-IDF+SVD (módulo 12).
TEXTOS_AJUSTE = [t for b in BENCHES for t in b.txt]
emb = None
for clave in ("multilingual-e5-small", "paraphrase-multilingual-MiniLM-L12-v2"):
    emb = cargar_embedder(clave)
    if emb is not None:
        break
if emb is None:
    emb = cargar_embedder("multilingual-e5-small", textos_ajuste=TEXTOS_AJUSTE)
for b in BENCHES:
    Ed, Eq = emb.encode_docs(b.txt), emb.encode_queries(b.qtxt)
    b.S["denso"] = Eq @ Ed.T
print(f"denso = {emb.clave} · real={emb.is_real}")

In [ ]:
import re, unicodedata
import scipy.sparse as sp
from collections import Counter

SUFIJOS = ("aciones", "acion", "amente", "mente", "ando", "iendo", "ing", "ed", "es", "s", "ar", "er", "ir")

def quita_acentos(t):
    n = unicodedata.normalize("NFD", t)
    return "".join(c for c in n if unicodedata.category(c) != "Mn")

def stem_ligero(t):
    """Stemmer de juguete (sufijos ES/EN). En producción: Snowball/Porter u otro analyzer del motor."""
    for s in SUFIJOS:
        if len(t) > len(s) + 3 and t.endswith(s):
            return t[:-len(s)]
    return t

def toks(texto, acentos=True, stem=False):
    t = texto.lower()
    if acentos:
        t = quita_acentos(t)
    out = re.findall(r"\w+", t)
    return [stem_ligero(x) for x in out] if stem else out

In [ ]:
import zlib
_BANCO_NGRAMAS = {}

def vec_ngrama(g, d):
    v = _BANCO_NGRAMAS.get((g, d))
    if v is None:
        v = np.random.default_rng(zlib.crc32(g.encode())).normal(size=d).astype(np.float32)
        _BANCO_NGRAMAS[(g, d)] = v
    return v

def vec_token(tok, d=64):
    """Embedding de JUGUETE de un token: suma de vectores aleatorios deterministas de sus 3-gramas de caracteres.
    Palabras que comparten letras (indexación/indexar) quedan cerca. No es semántico, pero hace los ejemplos reproducibles."""
    w = f"<{tok}>"
    grams = [w[i:i + 3] for i in range(len(w) - 2)] or [w]
    v = sum(vec_ngrama(g, d) for g in grams)
    return v / np.linalg.norm(v)

def multi_juguete(textos, d=64, max_tok=48):
    out = []
    for t in textos:
        tk = toks(t)[:max_tok] or ["vacio"]
        out.append(np.stack([vec_token(x, d) for x in tk]).astype(np.float32))
    return out

def maxsim(Q, D):
    """Q (nq,d), D (nd,d), filas normalizadas. S = suma_i max_j <q_i, d_j>."""
    return float((Q @ D.T).max(axis=1).sum())

In [ ]:
def minmax(S):
    lo, hi = S.min(1, keepdims=True), S.max(1, keepdims=True)
    return (S - lo) / np.maximum(hi - lo, 1e-9)

def zscore(S):
    return (S - S.mean(1, keepdims=True)) / np.maximum(S.std(1, keepdims=True), 1e-9)

In [ ]:
RERANK_MODELOS = ["cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"] + (["BAAI/bge-reranker-v2-m3"] if device == "cuda" else [])

class RerankerRespaldo:
    """⚠️ NO es un cross-encoder: puntúa pares con el MaxSim de juguete (solo para ejecutar el pipeline sin descargas)."""
    es_real = False
    def predict(self, pares, **kw):
        return np.array([maxsim(multi_juguete([q])[0], multi_juguete([d])[0]) / max(len(toks(q)), 1) for q, d in pares])

def carga_reranker():
    from sentence_transformers import CrossEncoder
    for nombre in RERANK_MODELOS:
        try:
            ce = CrossEncoder(nombre, device=device, max_length=256)
            ce.es_real = True
            return nombre, ce
        except Exception as e:
            print(f"[reranker] {nombre}: no disponible ({type(e).__name__}: {str(e)[:60]})")
    print("[reranker] usando reranker de respaldo (NO es un cross-encoder)")
    return "respaldo (MaxSim de juguete)", RerankerRespaldo()

ce_nombre, ce = carga_reranker()
NQ_REN = 30 if FAST_DEV_RUN else None          # nº de queries para medir el reranking (el cross-encoder es lento en CPU)

def reordena(b, S_primera, profundidad, nq=NQ_REN):
    """Devuelve (run, segundos_por_query): reordena el top-`profundidad` de S_primera con el cross-encoder."""
    run, t0 = {}, time.perf_counter()
    qs = list(enumerate(b.qs))[:nq]
    for i, q in qs:
        cand = np.argsort(-S_primera[i])[:profundidad]
        sc = np.asarray(ce.predict([(b.qtxt[i], b.txt[j]) for j in cand], batch_size=16, show_progress_bar=False)).ravel()
        run[q["qid"]] = [b.ids[cand[k]] for k in np.argsort(-sc)]
    return run, (time.perf_counter() - t0) / len(qs)

def ndcg_subconjunto(b, run):
    return float(np.mean([ndcg_at_k(run[q], b.qrels[q], 10) for q in run]))

## 6 · Paso 1 — BM25 desde cero (TODO 1)
Implementa `BM25.fit` y `BM25.scores` con una **matriz dispersa de pesos precomputada** (lección, sección 2). Variante de IDF: `lucene` → $\ln(1+\frac{N-n+.5}{n+.5})$; `okapi` → $\ln\frac{N-n+.5}{n+.5}$ con suelo $0.25\cdot\overline{\text{IDF}}$ para los negativos. Fórmula del peso de cada par (doc $i$, término $j$): `idf[j] * tf * (k1+1) / (tf + k1*(1 - b + b*dl[i]/avgdl))`.

<details><summary>🪜 Pista 1 (idea)</summary>
Cuenta <code>tf</code> con <code>Counter</code> y arma una <code>csr_matrix</code> (documentos × vocabulario). El peso de cada elemento no nulo depende de su fila (longitud del doc) y de su columna (IDF): calcula ambos arrays alineados con <code>tf.data</code>.
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>fila_de = np.repeat(np.arange(N), np.diff(tf.indptr))</code> · <code>w = tf.data*(k1+1)/(tf.data + k1*(1-b+b*dl[fila_de]/avgdl))</code> · <code>W = csr_matrix((w*idf[tf.indices], tf.indices, tf.indptr), shape=tf.shape).tocsc()</code> · <code>scores = W @ q</code> con <code>q</code> el vector de conteos de la query.
</details>

In [ ]:
class BM25:
    def __init__(self, k1=1.2, b=0.75, idf="lucene"):
        self.k1, self.b, self.idf_kind = k1, b, idf
    def fit(self, docs_tokens):
        # TODO 1.a: vocabulario, matriz tf (CSR), longitudes, IDF según self.idf_kind y matriz de pesos self.W
        raise NotImplementedError("TODO 1.a")
    def scores(self, q_tokens):
        # TODO 1.b: vector de conteos de la query y producto W @ q  (devuelve un array de n_docs)
        raise NotImplementedError("TODO 1.b")

def bm25_matriz(b, k1=1.2, bb=0.75, **analyzer):
    modelo = BM25(k1, bb).fit([toks(t, **analyzer) for t in b.txt])
    return np.stack([modelo.scores(toks(q, **analyzer)) for q in b.qtxt])

def paso1():
    from rank_bm25 import BM25Okapi
    docs_t = [toks(t) for t in B1.txt]
    ref, mio = BM25Okapi(docs_t, k1=1.2, b=0.75), BM25(idf="okapi").fit(docs_t)
    ok = all(np.allclose(ref.get_scores(toks(q)), mio.scores(toks(q))) for q in B1.qtxt[:10])
    print("coincide con rank_bm25:", ok)
    for b in BENCHES:
        b.S["bm25"] = bm25_matriz(b)
    return ok

try:
    R1_OK = paso1(); display(B1.resumen())
except NotImplementedError as e:
    R1_OK = False; print("⏳ pendiente:", e)

## 7 · Paso 2 — MaxSim vectorizado (TODO 2)
`maxsim(Q, D)` (ingenuo, dado) calcula un par. Implementa `maxsim_matriz(QV, DV)` que devuelve la matriz `(n_queries, n_docs)` **sin bucle por par**: concatena los tokens de todos los documentos, multiplica una vez por trozo de queries y toma máximos/sumas **por segmento**.

<details><summary>🪜 Pista 1 (idea)</summary>
<code>Qcat @ Dcat.T</code> da todas las similitudes token-token. El máximo por documento es un máximo sobre un <i>rango contiguo de columnas</i>; la suma por query, sobre un rango contiguo de filas.
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>np.maximum.reduceat(sims, inicios_docs, axis=1)</code> y después <code>np.add.reduceat(mx, inicios_queries, axis=0)</code>; los inicios salen de <code>np.cumsum([0] + longitudes[:-1])</code>. Procesa las queries en trozos de ~32 para no agotar RAM.
</details>

In [ ]:
def maxsim_matriz(QV, DV, trozo=32):
    # TODO 2: matriz (len(QV), len(DV)) de scores MaxSim, vectorizada con np.maximum.reduceat / np.add.reduceat
    raise NotImplementedError("TODO 2")

def paso2():
    for b in BENCHES:
        b.DV, b.QV = multi_juguete(b.txt), multi_juguete(b.qtxt)
        b.S["colbert-juguete"] = maxsim_matriz(b.QV, b.DV)
    rng = np.random.default_rng(0)
    pares = [(int(rng.integers(len(B1.QV))), int(rng.integers(len(B1.DV)))) for _ in range(20)]
    ok = all(np.isclose(B1.S["colbert-juguete"][i, j], maxsim(B1.QV[i], B1.DV[j]), atol=1e-4) for i, j in pares)
    print("coincide con la fórmula ingenua:", ok)
    return ok

try:
    R2_OK = paso2()
except NotImplementedError as e:
    R2_OK = False; print("⏳ pendiente:", e)

## 8 · Paso 3 — Fusión: RRF y combinación lineal (TODO 3) y α sin fuga (TODO 4)
- `rrf(Ss, k=60)`: para cada matriz de scores calcula el **rango** (1 = mejor) de cada documento por query y suma `1/(k+rango)`.
- `lineal(Ss, pesos, norm)`: suma ponderada de las matrices **normalizadas por query** (`minmax` o `zscore`, dadas).
- `alfa_cv(b, S1, S2, norm)`: elige α en la rejilla `np.linspace(0,1,21)` maximizando el nDCG medio sobre un pliegue de queries y evalúa en el **otro** (2 pliegues, `rng = np.random.default_rng(SEED)`, `idx = rng.permutation(n)`, `np.array_split(idx, 2)`); devuelve la media de los dos nDCG de validación. Usa `b.ndcg_q(S)` (nDCG por query).

<details><summary>🪜 Pista 1 (idea)</summary>
El rango de un documento es la posición en el orden descendente: <code>argsort(argsort(-S))+1</code> por fila. La fusión lineal normaliza cada fila antes de mezclar para que las escalas sean comparables.
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>rangos = np.argsort(np.argsort(-S, axis=1), axis=1) + 1</code> · <code>out += peso / (k + rangos)</code> · en <code>alfa_cv</code>: <code>mejor = max(grid, key=lambda a: b.ndcg_q(lineal([S1,S2],[a,1-a],norm))[tr].mean())</code>.
</details>

In [ ]:
def rrf(Ss, k=60, pesos=None):
    # TODO 3.a: fusión RRF de varias matrices de scores (nq, nd) -> matriz de scores fusionados
    raise NotImplementedError("TODO 3.a")

def lineal(Ss, pesos, norm=minmax):
    # TODO 3.b: sum_i peso_i * norm(S_i)
    raise NotImplementedError("TODO 3.b")

def alfa_cv(b, S1, S2, norm=minmax, grid=np.linspace(0, 1, 21)):
    # TODO 4: validación cruzada de 2 pliegues para elegir alfa; devuelve el nDCG@10 medio de validación
    raise NotImplementedError("TODO 4")

def paso3():
    r = rrf([np.array([[3.0, 1.0, 2.0]]), np.array([[1.0, 3.0, 2.0]])], k=60)
    # doc 0: rangos (1,3); doc 1: (3,1); doc 2: (2,2)
    ok_rrf = bool(np.isclose(r[0, 0], 1 / 61 + 1 / 63) and np.isclose(r[0, 2], 2 / 62))
    print("RRF ejemplo a mano:", ok_rrf)
    for b in BENCHES:
        b.S["rrf(bm25+denso)"] = rrf([b.S["bm25"], b.S["denso"]])
        b.S["rrf(bm25+denso+colbert)"] = rrf([b.S["bm25"], b.S["denso"], b.S["colbert-juguete"]])
        b.S["lineal α=0.5"] = lineal([b.S["bm25"], b.S["denso"]], [0.5, 0.5])
        b.cv = alfa_cv(b, b.S["bm25"], b.S["denso"])
        b.oraculo = max(b.ndcg(lineal([b.S["bm25"], b.S["denso"]], [a, 1 - a])) for a in np.linspace(0, 1, 21))
    return ok_rrf

try:
    R3_OK = paso3()
    for b in BENCHES:
        print(f"{b.nombre[:22]:>22}: α por CV → {b.cv:.3f} · α oráculo (mismas queries) → {b.oraculo:.3f}")
    display(pd.concat({b.nombre[:12]: b.resumen() for b in BENCHES}))
except (NotImplementedError, KeyError) as e:
    R3_OK = False; print("⏳ pendiente:", repr(e))

## 9 · Paso 4 — Pipeline con reranking (TODO 5) y presupuesto de latencia (TODO 6)
- `hibrido_rerank(b, S_fusion, K, nq)`: para cada query toma el top-`K` de `S_fusion`, puntúa los pares con `ce.predict` (cross-encoder real o de respaldo, ya cargado como `ce`) y devuelve `(run, ms_por_query)` donde `run[qid]` es la lista de **al menos 10** ids: los K reordenados seguidos del resto del ranking de `S_fusion` (así el nDCG@10 es comparable aunque `K<10`).
- `elige_K(latencias_ms, presupuesto_ms)`: dado un dict `{K: ms}` devuelve el **mayor K** cuya latencia sea ≤ presupuesto (o `None`).

<details><summary>🪜 Pista 1 (idea)</summary>
El reranking solo permuta los primeros K; el resto de la lista se mantiene en el orden de la fusión. Mide el tiempo con <code>time.perf_counter()</code> alrededor del bucle de queries.
</details>
<details><summary>🪜 Pista 2 (casi la API)</summary>
<code>cand = np.argsort(-S[i])</code> · <code>sc = ce.predict([(b.qtxt[i], b.txt[j]) for j in cand[:K]], batch_size=16, show_progress_bar=False)</code> · <code>orden = cand[:K][np.argsort(-np.asarray(sc).ravel())]</code> · <code>run[qid] = [b.ids[j] for j in list(orden) + list(cand[K:])][:max(10, K)]</code>.
</details>

In [ ]:
ce_nombre, ce = carga_reranker()
NQ_REN = 30 if FAST_DEV_RUN else None
PRESUPUESTO_MS = 200

def hibrido_rerank(b, S_fusion, K, nq=NQ_REN):
    # TODO 5: devuelve (run, ms_por_query) con el top-K reordenado por el cross-encoder `ce`
    raise NotImplementedError("TODO 5")

def elige_K(latencias_ms, presupuesto_ms=PRESUPUESTO_MS):
    # TODO 6: mayor K con latencia <= presupuesto (None si ninguno cabe)
    raise NotImplementedError("TODO 6")

def paso4():
    b = B1
    S_f = b.S["lineal α=0.5"]
    sub = {q["qid"] for q in b.qs[: (NQ_REN or len(b.qs))]}
    filas, lat = [], {}
    base = np.mean([ndcg_at_k(r, b.qrels[q], 10) for q, r in b.run(S_f, 10).items() if q in sub])
    filas.append(("sin rerank", float(base), 0.0))
    for K in (10, 20, 50):
        run, ms = hibrido_rerank(b, S_f, K)
        lat[K] = ms
        filas.append((f"rerank K={K}", float(np.mean([ndcg_at_k(r, b.qrels[q], 10) for q, r in run.items()])), ms))
    tabla = pd.DataFrame(filas, columns=["pipeline", "nDCG@10 (mismas queries)", "ms/query"])
    return tabla, lat, run

try:
    TABLA_REN, LAT, RUN_ULT = paso4()
    K_ELEGIDO = elige_K(LAT)
    display(TABLA_REN.round(3)); print("reranker:", ce_nombre, "· K elegido para", PRESUPUESTO_MS, "ms:", K_ELEGIDO)
except (NotImplementedError, KeyError) as e:
    TABLA_REN = LAT = RUN_ULT = K_ELEGIDO = None; print("⏳ pendiente:", repr(e))

## 10 · Autocomprobación de la rúbrica

In [ ]:
def autograder():
    ch = {}
    ch["R1 BM25 == rank_bm25"] = bool(R1_OK)
    ch["R2 maxsim_matriz == ingenuo"] = bool(R2_OK)
    ch["R3 RRF ejemplo a mano"] = bool(R3_OK)
    for etiqueta, b in (("R4 canónico", B1), ("R5 paráfrasis", B2)):
        try:
            indiv = max(b.ndcg(b.S[n]) for n in ("bm25", "denso", "colbert-juguete"))
            fus = max(b.cv, max(b.ndcg(b.S[n]) for n in ("rrf(bm25+denso)", "rrf(bm25+denso+colbert)", "lineal α=0.5")))
            ch[f"{etiqueta}: mejor fusión ≥ mejor individual − 0.03"] = bool(fus >= indiv - 0.03)
        except Exception:
            ch[f"{etiqueta}: mejor fusión ≥ mejor individual − 0.03"] = False
    try:
        ch["R6 CV sin fuga (cv ≤ oráculo)"] = all(b.cv <= b.oraculo + 1e-9 for b in BENCHES)
    except Exception:
        ch["R6 CV sin fuga (cv ≤ oráculo)"] = False
    try:
        ch["R7 pipeline devuelve ≥ 10 docs"] = all(len(v) >= 10 for v in RUN_ULT.values())
    except Exception:
        ch["R7 pipeline devuelve ≥ 10 docs"] = False
    try:
        ok = elige_K({10: 50, 20: 120, 50: 300}, 200) == 20 and elige_K({10: 500}, 200) is None
        ch["R8 elige_K respeta el presupuesto"] = bool(ok)
    except Exception:
        ch["R8 elige_K respeta el presupuesto"] = False
    for k, v in ch.items():
        print("✅" if v else "❌", k)
    return ch

_ = autograder()

## 11 · Tu conclusión (R9)
En ≤ 10 líneas, con números de tus tablas: ¿qué combinación de retrievers y qué fusión recomiendas? ¿Mejoró el reranker (con el modelo que tuvieras)? ¿Qué K cabe en 200 ms? ¿Qué riesgos ves (idioma, dominio, latencia p95, almacenamiento de la rama multi-vector)?

> *(Edita esta celda.)*

## 12 · Retos extra (nivel experto)
1. Añade la rama **sparse aprendido** o los tres modos de **BGE-M3** (lección, secciones 3 y 5) y repite las fusiones; ¿qué rama aporta más a la fusión en el banco de paráfrasis?
2. Implementa **MUVERA** (lección, sección 6) como primera etapa del multi-vector y mide recall del top-10 exacto vs número de candidatos.
3. Calcula **intervalos de confianza bootstrap** de la diferencia entre la mejor fusión y el mejor retriever individual.
4. Prueba un reranker **basado en LLM** (p. ej. Qwen3-Reranker) en una GPU y compara calidad/latencia con el cross-encoder.
5. Añade **filtros** (idioma, `lang` del documento) a la rama léxica y a la densa y mide qué pasa con el recall (anticipo del módulo 14).

## 13 · Reflexión (producción e hilo conductor)
- ¿Qué parte de este pipeline pondrías en el índice (offline) y cuál en la consulta (online)?
- Si el 30 % de las consultas son identificadores exactos, ¿qué peso darías a la rama léxica y cómo lo validarías?
- ¿Cómo monitorizarías la deriva de cada rama por separado? (módulo 18)
- ¿Cuánto crece el índice si activas late interaction para 10 M de documentos? (lección, sección 10)

---
# ⛔ SPOILER — intenta resolverlo primero
Las celdas siguientes **sobrescriben** tus funciones con la solución de referencia.

In [ ]:
class BM25:
    """BM25 con matriz de pesos precomputada. idf='lucene' o 'okapi' (como rank_bm25.BM25Okapi, epsilon=0.25)."""
    def __init__(self, k1=1.2, b=0.75, idf="lucene"):
        self.k1, self.b, self.idf_kind = k1, b, idf
    def fit(self, docs_tokens):
        self.vocab, filas, cols, vals = {}, [], [], []
        for i, tk in enumerate(docs_tokens):
            for t, c in Counter(tk).items():
                filas.append(i); cols.append(self.vocab.setdefault(t, len(self.vocab))); vals.append(c)
        N = len(docs_tokens)
        tf = sp.csr_matrix((vals, (filas, cols)), shape=(N, len(self.vocab)), dtype=np.float64)
        dl = np.array([len(t) for t in docs_tokens], dtype=float)
        avgdl = dl.mean()
        df = np.asarray((tf > 0).sum(0)).ravel()
        if self.idf_kind == "lucene":
            idf = np.log(1 + (N - df + 0.5) / (df + 0.5))
        else:
            idf = np.log(N - df + 0.5) - np.log(df + 0.5)
            idf = np.where(idf < 0, 0.25 * idf.mean(), idf)
        fila_de = np.repeat(np.arange(N), np.diff(tf.indptr))
        w = tf.data * (self.k1 + 1) / (tf.data + self.k1 * (1 - self.b + self.b * dl[fila_de] / avgdl))
        self.W = sp.csr_matrix((w * idf[tf.indices], tf.indices, tf.indptr), shape=tf.shape).tocsc()
        return self
    def scores(self, q_tokens):
        q = np.zeros(len(self.vocab))
        for t in q_tokens:
            if t in self.vocab:
                q[self.vocab[t]] += 1                 # los términos repetidos de la query suman varias veces
        return self.W @ q

In [ ]:
def maxsim_matriz(QV, DV, trozo=32):
    """Todos los pares (query, doc) sin bucles de Python por documento: concatena los tokens de los documentos,
    una sola multiplicación de matrices por trozo de queries y máximos por segmento con np.maximum.reduceat."""
    Dcat = np.concatenate(DV)
    ini_d = np.cumsum([0] + [len(x) for x in DV[:-1]])
    out = np.empty((len(QV), len(DV)), np.float32)
    for s in range(0, len(QV), trozo):
        qs = QV[s:s + trozo]
        Qcat = np.concatenate(qs)
        ini_q = np.cumsum([0] + [len(x) for x in qs[:-1]])
        mx = np.maximum.reduceat(Qcat @ Dcat.T, ini_d, axis=1)          # (Tq, nd): mejor token de cada doc
        out[s:s + len(qs)] = np.add.reduceat(mx, ini_q, axis=0)          # suma sobre los tokens de cada query
    return out

In [ ]:
def rrf(Ss, k=60, pesos=None):
    """Fusión RRF de varias matrices de scores (nq, nd). Devuelve una matriz de scores fusionados."""
    pesos = pesos or [1.0] * len(Ss)
    out = np.zeros_like(Ss[0], dtype=float)
    for S, w in zip(Ss, pesos):
        rangos = np.argsort(np.argsort(-S, axis=1), axis=1) + 1          # rango 1..nd de cada doc
        out += w / (k + rangos)
    return out

def lineal(Ss, pesos, norm=minmax):
    return sum(w * norm(S) for w, S in zip(pesos, Ss))

def alfa_cv(b, S1, S2, norm=minmax, grid=np.linspace(0, 1, 21)):
    """alfa elegido en un pliegue y evaluado en el otro (2 pliegues, semilla fija). Devuelve nDCG medio de validación."""
    rng = np.random.default_rng(SEED); idx = rng.permutation(len(b.qs)); mitades = np.array_split(idx, 2)
    res = []
    for tr, te in (mitades, mitades[::-1]):
        mejor = max(grid, key=lambda a: b.ndcg_q(lineal([S1, S2], [a, 1 - a], norm))[tr].mean())
        res.append(b.ndcg_q(lineal([S1, S2], [mejor, 1 - mejor], norm))[te].mean())
    return float(np.mean(res))

In [ ]:
def hibrido_rerank(b, S_fusion, K, nq=NQ_REN):
    run, t0 = {}, time.perf_counter()
    qs = list(enumerate(b.qs))[:nq]
    for i, q in qs:
        cand = np.argsort(-S_fusion[i])
        sc = np.asarray(ce.predict([(b.qtxt[i], b.txt[j]) for j in cand[:K]], batch_size=16, show_progress_bar=False)).ravel()
        orden = cand[:K][np.argsort(-sc)]
        run[q["qid"]] = [b.ids[j] for j in list(orden) + list(cand[K:])][:max(10, K)]
    return run, (time.perf_counter() - t0) / len(qs) * 1e3

def elige_K(latencias_ms, presupuesto_ms=PRESUPUESTO_MS):
    ok = [K for K, ms in latencias_ms.items() if ms <= presupuesto_ms]
    return max(ok) if ok else None

R1_OK = paso1()
R2_OK = paso2()
R3_OK = paso3()
for b in BENCHES:
    print(f"{b.nombre[:22]:>22}: α por CV → {b.cv:.3f} · oráculo → {b.oraculo:.3f}")
display(pd.concat({b.nombre[:12]: b.resumen() for b in BENCHES}))
TABLA_REN, LAT, RUN_ULT = paso4()
K_ELEGIDO = elige_K(LAT)
display(TABLA_REN.round(3)); print("reranker:", ce_nombre, "· K elegido:", K_ELEGIDO)
_ = autograder()

### Conclusión de referencia (ejemplo)
En el entorno de pruebas (sin modelos reales) BM25 domina el banco canónico porque las queries comparten vocabulario con los documentos, y la fusión no lo supera pero queda a menos de 0.03 de nDCG@10; en el banco de paráfrasis todas las ramas léxicas sufren y la fusión rinde como la mejor rama. El α elegido por validación cruzada es ≤ al α oráculo (el oráculo está sesgado). El reranking de respaldo no es un cross-encoder: su calidad no es informativa, pero su latencia crece ≈ lineal en K, y K se elige desde el presupuesto de 200 ms. Con modelos reales esperarías que denso y late interaction aporten en paráfrasis y entre idiomas y que el cross-encoder suba el nDCG@10 en el banco de paráfrasis; **verifícalo con tus números**.